# 🏋️ تدريب وكيل الشركة بالتعلم المعزز (مهارة: دخول بوابة + نفاذ + قراءة الجدول)

**قبل البدء:** من القائمة اختر *Runtime ← Change runtime type ← T4 GPU* ثم شغّل الخلايا **بالترتيب** (زر ▶ لكل خلية).

**ماذا يحدث:** النموذج (Qwen صغير) يتدرب في بيئة محاكاة لبوابة حكومية وهمية. يحصل على مكافأة تلقائية: درجة كاملة إذا حسب الأيام المتبقية لكل عامل صح، وخصم إذا خمّن كلمة السر أو ضغط زر حساس أو سلّم نتيجة خاطئة. لا بيانات حقيقية ولا مواقع حقيقية.

**تنبيه صادق:** واجهة مكتبة ART تتغير بين الإصدارات، وهذا الدفتر ما جُرّب على GPU. إذا ظهر خطأ في خلية، اضغط *Explain error / Fix with Gemini* (يظهر تحت الخطأ في Colab) أو انسخ الخطأ وأرسله لي.

In [ ]:
# 1) تثبيت المكتبات (تأخذ عدة دقائق؛ قد يطلب إعادة تشغيل الجلسة: اختر Restart ثم أعد تشغيل الخلية)
!pip install -q "openpipe-art[backend]"
!git clone -q https://github.com/khd98me-svg/org-agents.git || (cd org-agents && git pull -q)
import sys; sys.path.insert(0, "org-agents/rl")
import portal_env as P, rollout_core as R
print("البيئة جاهزة. مثال صفحة:\n", P.PortalEnv(1).snapshot())

In [ ]:
# 2) الإعدادات (غيّر الأرقام لو الذاكرة لم تكفِ: قلّل ROLLOUTS_PER_SCENARIO أو غيّر النموذج إلى "Qwen/Qwen2.5-1.5B-Instruct")
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
PROJECT, NAME = "company-agents", "portal-001"
STEPS = 30                 # عدد خطوات التدريب (كل خطوة تأخذ عدة دقائق)
SCENARIOS_PER_STEP = 8     # عدد الحالات المختلفة في كل خطوة
ROLLOUTS_PER_SCENARIO = 6  # محاولات لكل حالة (GRPO يقارن بينها)
EVAL_SEEDS = list(range(5000, 5016))

In [ ]:
# 3) تسجيل النموذج والخادم المحلي
import art, statistics
from art.local import LocalBackend
backend = LocalBackend()
model = art.TrainableModel(name=NAME, project=PROJECT, base_model=BASE_MODEL)
await model.register(backend)

In [ ]:
# 4) دالة الحلقة: تلف البيئة بواجهة ART (كل حلقة مستقلة، وتعمل عدة حلقات بالتوازي)
import asyncio

async def rollout(model, seed):
    client = model.openai_client()
    env = P.PortalEnv(seed)
    traj = art.Trajectory(
        messages_and_choices=[{"role": "system", "content": P.SYSTEM},
                              {"role": "user", "content": P.TASK + "\n\nالصفحة الحالية:\n" + env.snapshot()}],
        reward=0.0)
    for _ in range(P.MAX_STEPS):
        r = await client.chat.completions.create(model=model.inference_model_name, messages=traj.messages(),
                                                 max_completion_tokens=300, temperature=1.0, timeout=120)
        choice = r.choices[0]
        traj.messages_and_choices.append(choice)          # رد النموذج كما هو (مطلوب للتدريب)
        obs, done = env.step(choice.message.content or "")
        if done:
            break
        traj.messages_and_choices.append({"role": "user", "content": obs})
    traj.reward = env.reward()
    try:
        traj.metrics["steps"] = env.steps
    except Exception:
        pass
    return traj

async def evaluate(seeds):
    trs = await asyncio.gather(*[rollout(model, s) for s in seeds])
    return statistics.mean(t.reward for t in trs)


In [ ]:
# 5) قياس قبل التدريب (النموذج الخام)
before = await evaluate(EVAL_SEEDS)
print("متوسط المكافأة قبل التدريب: %.3f  (الحل اليدوي ≈ 1.21، التسليم الفارغ = -0.20)" % before)

In [ ]:
# 6) التدريب. كل خطوة تُحفظ تلقائياً، فلو انقطعت الجلسة شغّل الخلايا 3 ثم 4 ثم هذه وتكمل من آخر خطوة.
start = await model.get_step()
for i in range(start, STEPS):
    seeds = [10_000 + i * SCENARIOS_PER_STEP + k for k in range(SCENARIOS_PER_STEP)]
    groups = await art.gather_trajectory_groups(
        art.TrajectoryGroup(rollout(model, s) for _ in range(ROLLOUTS_PER_SCENARIO)) for s in seeds
    )
    mean_r = statistics.mean(t.reward for g in groups for t in g.trajectories)
    await model.train(groups)
    print("خطوة %d/%d | متوسط مكافأة التدريب %.3f" % (i + 1, STEPS, mean_r))

In [ ]:
# 7) قياس بعد التدريب على حالات لم يرها النموذج
after = await evaluate(EVAL_SEEDS)
print("قبل: %.3f  →  بعد: %.3f" % (before, after))
print("نجح التدريب فعلاً" if after > before + 0.15 else "التحسن ضعيف: زد STEPS أو جرّب البداية الدافئة (demos) في README")

In [ ]:
# 8) حفظ النتيجة في Google Drive (مجلد التدريب .art فيه النموذج المتدرب)
from google.colab import drive
drive.mount("/content/drive")
!cp -r .art /content/drive/MyDrive/company-rl-art && echo "تم النسخ إلى MyDrive/company-rl-art"